In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


In [5]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster


# ============================================================
# PREPARAÇÃO DOS DADOS
# ============================================================

# Verificar as primeiras linhas
print(df_mapa.head())

# Remover imóveis sem latitude ou longitude
df_mapa = df_mapa.dropna(subset=['latitude', 'longitude']).copy()

# Calcular a coordenada média dos imóveis
latitude_media = df_mapa['latitude'].mean()
longitude_media = df_mapa['longitude'].mean()

print("Latitude média:", latitude_media)
print("Longitude média:", longitude_media)


# ============================================================
# PARTE 1 - INICIALIZAÇÃO E MARCADORES BÁSICOS
# ============================================================

# 1. Criar mapa base
mapa_base = folium.Map(
    location=[latitude_media, longitude_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)


# 2. Adicionar marcadores para os 5 primeiros imóveis

for _, imovel in df_mapa.head(5).iterrows():

    popup_texto = f"""
    <b>Tipo do imóvel:</b> {imovel['tipo']}<br>
    <b>Valor de venda:</b> R$ {imovel['valor_venda']:,.2f}
    """

    folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=folium.Popup(
            popup_texto,
            max_width=300
        )
    ).add_to(mapa_base)


# Exibir o primeiro mapa
mapa_base


# ============================================================
# PARTE 2 - CUSTOMIZAÇÃO VISUAL COM CIRCLEMARKER
# ============================================================

# 3. Criar um novo mapa
mapa_circulos = folium.Map(
    location=[latitude_media, longitude_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)


# 4. Adicionar CircleMarker para todos os imóveis

for _, imovel in df_mapa.iterrows():

    # Definir a cor de acordo com a cidade
    if imovel['cidade'] == 'Nova Iguaçu':
        cor = 'blue'

    elif imovel['cidade'] == 'Queimados':
        cor = 'orange'

    else:
        # Cor para outras cidades
        cor = 'gray'

    folium.CircleMarker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes'
    ).add_to(mapa_circulos)


# Exibir o segundo mapa
mapa_circulos


# ============================================================
# PARTE 3 - AGRUPAMENTO INTELIGENTE (CLUSTERING)
# ============================================================

# 5. Criar um terceiro mapa
mapa_cluster = folium.Map(
    location=[latitude_media, longitude_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)


# Criar o MarkerCluster
cluster = MarkerCluster().add_to(mapa_cluster)


# 6. Definir cores de acordo com o tipo do imóvel

cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}


# Adicionar todos os imóveis ao cluster
for _, imovel in df_mapa.iterrows():

    tipo = imovel['tipo']

    # Se o tipo não estiver no dicionário, usar preto
    cor_icone = cores_tipo.get(tipo, 'black')

    popup_texto = f"""
    <b>Tipo:</b> {tipo}<br>
    <b>Valor:</b> R$ {imovel['valor_venda']:,.2f}<br>
    <b>Cidade:</b> {imovel['cidade']}
    """

    marcador = folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=folium.Popup(
            popup_texto,
            max_width=300
        ),
        icon=folium.Icon(
            color=cor_icone,
            icon='home',
            prefix='fa'
        )
    )

    # Adicionar marcador ao cluster
    marcador.add_to(cluster)


# ============================================================
# 7. SALVAR O MAPA FINAL
# ============================================================

mapa_cluster.save('mapa_imoveis_baixada.html')

print("Mapa salvo com sucesso!")
print("Arquivo: mapa_imoveis_baixada.html")


# Exibir o mapa final
mapa_cluster

   id_imovel       cidade  valor_venda         tipo   latitude  longitude
0          1    Queimados    613765.60         Casa -22.714426 -43.571388
1          2  Nova Iguaçu    368197.75      Terreno -22.737554 -43.439882
2          3  Nova Iguaçu    514047.61  Apartamento -22.732235 -43.470753
3          4  Nova Iguaçu    532697.20      Terreno -22.766920 -43.478809
4          5    Queimados    279398.12         Casa -22.731598 -43.573369
Latitude média: -22.736785025907697
Longitude média: -43.50738358758272
Mapa salvo com sucesso!
Arquivo: mapa_imoveis_baixada.html
